# Retrival

In diesem Schritt geht es um das Query an die Datenbank um uns eine reihe von ähnlichen Chunks zu erhalten. 

In [ ]:
import json
import chromadb
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('deepset/gbert-large')
client = chromadb.PersistentClient(path="../data/vector_store")
collection = client.get_collection('ProduktRAG')

## Retrival

Alle Fragen nacheinander an die DB senden... zuvor natürlich das Embedding der Frage.

In [ ]:
with open('../data/tests/specs_question.json', 'r') as f:
    quests = json.load(f)

results = []

for quest in tqdm(quests, total=len(quests)):
    embedding = model.encode(quest['question']).tolist()
    result = collection.query(query_embeddings=[embedding], n_results=10)

    results.append(result)

## Evaluation

Geprüft werden jetzt

* Recall@k: Wurde der korrekte Chunk gefunden?
* Mean Reciprocal Rank (MRR): Auf welcher Postition wurde der korrekte Chunk durchschnittlich gefunden?
* Precision@k: Wie relevant sind die gefundenen Chunks?

Für die Fragen mit einem Chunk als Antwort wird hier Recall@k und MRR verwendet. Precision@k ist hingegen für jene Fragen wichtig, die mehr als nur einen Chunk zur Beantwortung benötigen.

Schlechte Retrivales werden nochmal untersucht um zu prüfen, wo es Verbesserungspotential gibt.

In [ ]:
print(f"Anzahl Queries: {len(quests)}")
print(f"Anzahl Results: {len(results)}")
print()

recall_at_1, recall_at_3, recall_at_5, recall_at_10 = [], [], [], []
mrr_scores = []
badies = []

for i in range(len(quests)):
    expected_id = quests[i]['chunk_id']
    retrieved_ids = results[i]['ids'][0]

    # Recall
    recall_at_1.append(1 if expected_id in retrieved_ids[:1] else 0)
    recall_at_3.append(1 if expected_id in retrieved_ids[:3] else 0)
    recall_at_5.append(1 if expected_id in retrieved_ids[:5] else 0)
    recall_at_10.append(1 if expected_id in retrieved_ids[:10] else 0)

    # MRR
    if expected_id in retrieved_ids:
        position = retrieved_ids.index(expected_id) + 1
        mrr_scores.append(1.0 / position)
    else:
        mrr_scores.append(0.0)
    
    # Get bad retrivals
    if recall_at_5[i] == 0:
        badies.append({
            'query': quests[i]['question'],
            'answers': results[i]['documents']
        })

print(f"Recall@1:  {sum(recall_at_1) / len(recall_at_1):.2%}")
print(f"Recall@3:  {sum(recall_at_3) / len(recall_at_3):.2%}")
print(f"Recall@5:  {sum(recall_at_5) / len(recall_at_5):.2%}")
print(f"Recall@10: {sum(recall_at_10) / len(recall_at_10):.2%}")
print()
print(f"MRR:       {sum(mrr_scores) / len(mrr_scores):.3f}")
print()

print(json.dumps(badies, indent=2, ensure_ascii=False))
